# Day 7 — Dynamic Memory + STL

---

## 7.1 Dynamic Memory Allocation

Dynamic memory allocation allows programs to request memory at **runtime** rather than compile time. In C++, this is done using `new` and `delete`.

### Stack vs Heap Memory

| Aspect | Stack | Heap |
|---|---|---|
| **Allocation** | Automatic (function call) | Manual (`new` / `delete`) |
| **Size** | Small (typically ~1-8 MB) | Large (RAM / virtual memory) |
| **Speed** | Fast (single instruction) | Slower (system call overhead) |
| **Lifetime** | Function scope | Until explicitly deleted |
| **Management** | Compiler handles it | Programmer must handle |

### `new` and `delete` Operators

In [ ]:
#include <iostream>
using namespace std;

int main() {
    // Single variable allocation
    int* p = new int;        // Allocate memory for one int
    *p = 42;                 // Assign value
    cout << "Value: " << *p << endl;
    delete p;                // Free memory
    p = nullptr;             // Good practice: nullify pointer
    
    // Initialization during allocation
    int* q = new int(100);   // Allocate and initialize to 100
    cout << "Value: " << *q << endl;
    delete q;
    
    return 0;
}

### `new[]` and `delete[]` for Arrays

In [ ]:
#include <iostream>
using namespace std;

int main() {
    // Dynamic array allocation
    int size;
    cout << "Enter array size: ";
    cin >> size;
    
    int* arr = new int[size];   // Allocate array on heap
    
    // Use the array
    for (int i = 0; i < size; i++) {
        arr[i] = (i + 1) * 10;
    }
    
    cout << "Array elements: ";
    for (int i = 0; i < size; i++) {
        cout << arr[i] << " ";
    }
    cout << endl;
    
    delete[] arr;   // Must use delete[] for arrays!
    arr = nullptr;
    
    return 0;
}

**Sample Run:**

In [ ]:
Enter array size: 5
Array elements: 10 20 30 40 50

> **Critical rule:** Use `delete` for `new` and `delete[]` for `new[]`. Mismatching causes **undefined behavior**.

---

### Memory Leaks

A **memory leak** occurs when dynamically allocated memory is never deallocated.

In [ ]:
// BAD — Memory leak!
void leakExample() {
    int* data = new int[1000];
    // ... do some work ...
    // Forgot to delete[] data — 4000 bytes leaked!
}

// GOOD — Proper cleanup
void safeExample() {
    int* data = new int[1000];
    // ... do some work ...
    delete[] data;  // Memory freed
}

### Consequences of Memory Leaks

- Program consumes increasing memory over time
- System performance degrades
- Program may crash (out-of-memory)
- In long-running processes (servers), leaks are catastrophic

---

## 7.2 Pointers to Objects

### Arrow Operator (->)

When accessing members of an object **through a pointer**, use the arrow operator `->`:

In [ ]:
#include <iostream>
#include <string>
using namespace std;

class Student {
    public:
        string name;
        int age;
        
        Student(string n, int a) : name(n), age(a) {}
        
        void display() {
            cout << name << " (" << age << " years old)" << endl;
        }
};

int main() {
    // Object on heap
    Student* s1 = new Student("Arjun", 20);
    
    // Access members via arrow operator
    cout << "Name: " << s1->name << endl;     // Equivalent to (*s1).name
    cout << "Age: " << s1->age << endl;         // Equivalent to (*s1).age
    s1->display();                               // Equivalent to (*s1).display()
    
    // Alternative: dereference first, then use dot
    (*s1).display();
    
    delete s1;
    return 0;
}

**Output:**

In [ ]:
Name: Arjun
Age: 20
Arjun (20 years old)
Arjun (20 years old)

---

## 7.3 Array of Objects using Pointers

In [ ]:
#include <iostream>
#include <string>
using namespace std;

class Product {
    private:
        int id;
        string name;
        double price;
    
    public:
        Product() : id(0), name("N/A"), price(0.0) {}
        
        Product(int i, string n, double p) : id(i), name(n), price(p) {}
        
        void display() const {
            cout << "ID: " << id << " | " << name << " | $" << price << endl;
        }
};

int main() {
    // Dynamic array of objects
    int count;
    cout << "How many products? ";
    cin >> count;
    
    Product* inventory = new Product[count];
    
    // Initialize each element
    for (int i = 0; i < count; i++) {
        inventory[i] = Product(100 + i, "Product-" + to_string(i + 1), (i + 1) * 10.0);
    }
    
    // Display all
    cout << "\nInventory:" << endl;
    for (int i = 0; i < count; i++) {
        inventory[i].display();  // Dot operator (array syntax)
    }
    
    delete[] inventory;
    return 0;
}

**Sample Run:**

In [ ]:
How many products? 3

Inventory:
ID: 100 | Product-1 | $10
ID: 101 | Product-2 | $20
ID: 102 | Product-3 | $30

---

## 7.4 Introduction to Templates

Templates allow writing **generic** code that works with any data type. The compiler generates type-specific code at compile time.

### 7.4.1 Function Templates

In [ ]:
#include <iostream>
#include <string>
using namespace std;

// Template function — works with ANY type
template <typename T>
T getMax(T a, T b) {
    return (a > b) ? a : b;
}

// Template with multiple type parameters
template <typename T, typename U>
auto add(T a, U b) -> decltype(a + b) {
    return a + b;
}

int main() {
    // Compiler deduces type from arguments
    cout << "Max of 10 and 20: " << getMax(10, 20) << endl;           // int
    cout << "Max of 3.14 and 2.71: " << getMax(3.14, 2.71) << endl;  // double
    cout << "Max of 'a' and 'z': " << getMax('a', 'z') << endl;      // char
    
    // Explicit type specification
    cout << "Max (explicit double): " << getMax<double>(5, 7.5) << endl;
    
    // Multiple type parameters
    cout << "Add int + double: " << add(10, 3.5) << endl;
    cout << "Add string + string: " << add(to_string(42), to_string(100)) << endl;
    
    return 0;
}


**Output:**

In [ ]:
Max of 10 and 20: 20
Max of 3.14 and 2.71: 3.14
Max of 'a' and 'z': z
Max (explicit double): 7.5
Add int + double: 13.5
Add int + string: 42100

### 7.4.2 Class Templates

In [ ]:
#include <iostream>
#include <string>
using namespace std;

// Generic Pair class
template <typename T>
class Pair {
    private:
        T first;
        T second;
    
    public:
        Pair(T f, T s) : first(f), second(s) {}
        
        T getFirst() const { return first; }
        T getSecond() const { return second; }
        
        T getMax() const { return (first > second) ? first : second; }
        
        void display() const {
            cout << "(" << first << ", " << second << ")" << endl;
        }
};

// Template with multiple types
template <typename T, typename U>
class KeyValue {
    private:
        T key;
        U value;
    
    public:
        KeyValue(T k, U v) : key(k), value(v) {}
        
        void display() const {
            cout << "[" << key << "] => " << value << endl;
        }
};

int main() {
    // Pair with different types
    Pair<int> p1(10, 20);
    Pair<double> p2(3.14, 2.71);
    Pair<string> p3("Apple", "Orange");
    
    cout << "Int pair: "; p1.display();
    cout << "Max: " << p1.getMax() << endl;
    
    cout << "Double pair: "; p2.display();
    cout << "Max: " << p2.getMax() << endl;
    
    cout << "String pair: "; p3.display();
    cout << "Max: " << p3.getMax() << endl;
    
    // KeyValue with different types for key and value
    KeyValue<int, string> kv(101, "Arjun");
    KeyValue<string, double> kv2("Pi", 3.14159);
    
    cout << "\nKey-Value pairs:" << endl;
    kv.display();
    kv2.display();
    
    return 0;
}

**Output:**

In [ ]:
Int pair: (10, 20)
Max: 20
Double pair: (3.14, 2.71)
Max: 3.14
String pair: (Apple, Orange)
Max: Orange

Key-Value pairs:
[101] => Arjun
[Pi] => 3.14159

---

## 7.5 Introduction to STL (Standard Template Library)

The **STL** is a powerful library of generic **containers**, **algorithms**, and **iterators**.

### STL Components

In [ ]:
STL
├── Containers       (store data)
│   ├── Sequence:   vector, list, deque, array, forward_list
│   ├── Associative: set, multiset, map, multimap
│   └── Unordered:  unordered_set, unordered_map
├── Algorithms      (process data)
│   ├── Sorting:    sort, stable_sort, partial_sort
│   ├── Searching:  find, binary_search, lower_bound
│   └── Modifying:  copy, remove, replace, transform
└── Iterators       (bridge between containers and algorithms)
    ├── Input, Output
    ├── Forward, Bidirectional
    └── Random Access

### 7.5.1 `vector` — Dynamic Array

In [ ]:
#include <iostream>
#include <vector>
using namespace std;

int main() {
    // Declaration
    vector<int> vec1;                // Empty vector
    vector<int> vec2(5, 10);         // 5 elements, all = 10
    vector<int> vec3 = {1, 2, 3, 4, 5};  // Initializer list
    
    cout << "vec3: ";
    for (int x : vec3) cout << x << " ";
    cout << endl;
    
    // Common operations
    vec3.push_back(6);          // Add at end: O(1) amortized
    vec3.push_back(7);
    
    cout << "After push_back: ";
    for (int x : vec3) cout << x << " ";
    cout << endl;
    
    cout << "Size: " << vec3.size() << endl;
    cout << "Capacity: " << vec3.capacity() << endl;
    cout << "First: " << vec3.front() << endl;
    cout << "Last: " << vec3.back() << endl;
    
    vec3.pop_back();            // Remove last element
    cout << "After pop_back: ";
    for (int x : vec3) cout << x << " ";
    cout << endl;
    
    // Insert at position
    vec3.insert(vec3.begin() + 2, 99);  // Insert 99 at index 2
    cout << "After insert: ";
    for (int x : vec3) cout << x << " ";
    cout << endl;
    
    // Remove at position
    vec3.erase(vec3.begin() + 1);  // Remove element at index 1
    cout << "After erase: ";
    for (int x : vec3) cout << x << " ";
    cout << endl;
    
    // Clearing
    vec3.clear();
    cout << "After clear, size: " << vec3.size() << endl;
    
    return 0;
}

**Output:**

In [ ]:
vec3: 1 2 3 4 5 
After push_back: 1 2 3 4 5 6 7 
Size: 7
Capacity: 10
First: 1
Last: 7
After pop_back: 1 2 3 4 5 6 
After insert: 1 2 99 3 4 5 6 
After erase: 1 99 3 4 5 6 
After clear, size: 0

### 7.5.2 `list` — Doubly Linked List

In [ ]:
#include <iostream>
#include <list>
using namespace std;

int main() {
    list<int> myList = {10, 20, 30, 40, 50};
    
    // Add elements
    myList.push_front(5);    // O(1) — advantage over vector
    myList.push_back(60);    // O(1)
    
    cout << "List: ";
    for (int x : myList) cout << x << " ";
    cout << endl;
    
    // Remove elements
    myList.pop_front();      // O(1)
    myList.pop_back();       // O(1)
    
    cout << "After pop_front and pop_back: ";
    for (int x : myList) cout << x << " ";
    cout << endl;
    
    // Insert and erase (O(n) for traversal)
    auto it = myList.begin();
    advance(it, 2);          // Move iterator to position 2
    myList.insert(it, 25);   // Insert at position 2
    
    cout << "After insert at index 2: ";
    for (int x : myList) cout << x << " ";
    cout << endl;
    
    // Sort
    myList.sort();
    cout << "Sorted: ";
    for (int x : myList) cout << x << " ";
    cout << endl;
    
    return 0;
}

**Output:**

In [ ]:
List: 5 10 20 30 40 50 60 
After pop_front and pop_back: 10 20 30 40 50 
After insert at index 2: 10 20 25 30 40 50 
Sorted: 10 20 25 30 40 50

### 7.5.3 `stack` — LIFO Structure

In [ ]:
#include <iostream>
#include <stack>
using namespace std;

int main() {
    stack<int> s;
    
    // Push elements
    s.push(10);
    s.push(20);
    s.push(30);
    s.push(40);  // Top element
    
    cout << "Stack size: " << s.size() << endl;
    cout << "Top element: " << s.top() << endl;  // 40
    
    cout << "Popping all elements: ";
    while (!s.empty()) {
        cout << s.top() << " ";  // View top
        s.pop();                  // Remove top
    }
    cout << endl;
    
    // Stack is now empty
    cout << "Is empty? " << (s.empty() ? "Yes" : "No") << endl;
    
    return 0;
}

**Output:**

In [ ]:
Stack size: 4
Top element: 40
Popping all elements: 40 30 20 10 
Is empty? Yes

### 7.5.4 `queue` — FIFO Structure

In [ ]:
#include <iostream>
#include <queue>
using namespace std;

int main() {
    queue<string> q;
    
    // Enqueue elements
    q.push("Task 1");
    q.push("Task 2");
    q.push("Task 3");
    
    cout << "Queue size: " << q.size() << endl;
    cout << "Front: " << q.front() << endl;  // Task 1
    cout << "Back: " << q.back() << endl;    // Task 3
    
    cout << "Processing queue: ";
    while (!q.empty()) {
        cout << "[" << q.front() << "] ";  // View front
        q.pop();                            // Remove front
    }
    cout << endl;
    
    return 0;
}

**Output:**

In [ ]:
Queue size: 3
Front: Task 1
Back: Task 3
Processing queue: [Task 1] [Task 2] [Task 3]

---

## 7.6 Iterators and Basic Algorithms

### Iterators

Iterators are objects that **point to elements** in containers, similar to pointers.

In [ ]:
#include <iostream>
#include <vector>
#include <algorithm>
using namespace std;

int main() {
    vector<int> vec = {40, 10, 30, 20, 50};
    
    // Getting iterators
    auto it_begin = vec.begin();   // Points to first element
    auto it_end = vec.end();       // Points to one-past-last
    
    // Iterator traversal
    cout << "Forward: ";
    for (auto it = vec.begin(); it != vec.end(); ++it) {
        cout << *it << " ";  // Dereference iterator like a pointer
    }
    cout << endl;
    
    // Reverse traversal
    cout << "Reverse: ";
    for (auto it = vec.rbegin(); it != vec.rend(); ++it) {
        cout << *it << " ";
    }
    cout << endl;
    
    // Range-based for loop (uses iterators internally)
    cout << "Range-for: ";
    for (int x : vec) {
        cout << x << " ";
    }
    cout << endl;
    
    return 0;
}

**Output:**

In [ ]:
Forward: 40 10 30 20 50
Reverse: 50 20 30 10 40
Range-for: 40 10 30 20 50

### Basic Algorithms

In [ ]:
#include <iostream>
#include <vector>
#include <algorithm>
#include <numeric>  // For accumulate
using namespace std;

int main() {
    vector<int> vec = {40, 10, 30, 20, 50};
    
    // 1. sort()
    vector<int> sorted = vec;
    sort(sorted.begin(), sorted.end());
    cout << "Sorted: ";
    for (int x : sorted) cout << x << " ";
    cout << endl;
    
    // Sort in descending order
    sort(sorted.begin(), sorted.end(), greater<int>());
    cout << "Descending: ";
    for (int x : sorted) cout << x << " ";
    cout << endl;
    
    // 2. find() — returns iterator to element (or end() if not found)
    auto it = find(vec.begin(), vec.end(), 30);
    if (it != vec.end()) {
        cout << "Found 30 at position: " << (it - vec.begin()) << endl;
    } else {
        cout << "30 not found" << endl;
    }
    
    // 3. count() — counts occurrences
    vector<int> data = {1, 2, 2, 3, 2, 4, 2, 5};
    int twos = count(data.begin(), data.end(), 2);
    cout << "Number of 2s: " << twos << endl;
    
    // 4. binary_search() — requires sorted range
    if (binary_search(sorted.begin(), sorted.end(), 30)) {
        cout << "30 exists in sorted array" << endl;
    }
    
    // 5. accumulate() — sum of elements
    int sum = accumulate(vec.begin(), vec.end(), 0);
    cout << "Sum of elements: " << sum << endl;
    
    // 6. min_element() / max_element()
    auto minIt = min_element(vec.begin(), vec.end());
    auto maxIt = max_element(vec.begin(), vec.end());
    cout << "Min: " << *minIt << ", Max: " << *maxIt << endl;
    
    // 7. reverse()
    vector<int> rev = vec;
    reverse(rev.begin(), rev.end());
    cout << "Reversed: ";
    for (int x : rev) cout << x << " ";
    cout << endl;
    
    return 0;
}

**Output:**

In [ ]:
Sorted: 10 20 30 40 50
Descending: 50 40 30 20 10
Found 30 at position: 2
Number of 2s: 4
30 exists in sorted array
Sum of elements: 150
Min: 10, Max: 50
Reversed: 50 20 30 10 40

---

## Coding Examples

### Example 1 — Dynamic Student Records with Templates

In [ ]:
#include <iostream>
#include <vector>
#include <string>
#include <algorithm>
using namespace std;

template <typename T>
class ScoreManager {
    private:
        vector<T> scores;
    
    public:
        void addScore(T score) {
            scores.push_back(score);
        }
        
        T getAverage() {
            if (scores.empty()) return T();
            T sum = 0;
            for (T s : scores) sum += s;
            return sum / scores.size();
        }
        
        T getHighest() {
            return *max_element(scores.begin(), scores.end());
        }
        
        T getLowest() {
            return *min_element(scores.begin(), scores.end());
        }
        
        void displayAll() {
            cout << "Scores: ";
            for (T s : scores) cout << s << " ";
            cout << endl;
        }
        
        int getCount() { return scores.size(); }
};

int main() {
    // Integer scores
    ScoreManager<int> mathScores;
    mathScores.addScore(85);
    mathScores.addScore(92);
    mathScores.addScore(78);
    mathScores.addScore(95);
    mathScores.addScore(88);
    
    cout << "=== Math Scores (int) ===" << endl;
    mathScores.displayAll();
    cout << "Count: " << mathScores.getCount() << endl;
    cout << "Average: " << mathScores.getAverage() << endl;
    cout << "Highest: " << mathScores.getHighest() << endl;
    cout << "Lowest: " << mathScores.getLowest() << endl;
    
    // Double scores
    ScoreManager<double> scienceScores;
    scienceScores.addScore(88.5);
    scienceScores.addScore(91.2);
    scienceScores.addScore(76.8);
    
    cout << "\n=== Science Scores (double) ===" << endl;
    scienceScores.displayAll();
    cout << "Average: " << scienceScores.getAverage() << endl;
    
    return 0;
}

**Output:**

In [ ]:
=== Math Scores (int) ===
Scores: 85 92 78 95 88 
Count: 5
Average: 87
Highest: 95
Lowest: 78

=== Science Scores (double) ===
Scores: 88.5 91.2 76.8 
Average: 85.5

---

### Example 2 — STL Container Comparison

In [ ]:
#include <iostream>
#include <vector>
#include <list>
#include <stack>
#include <queue>
#include <chrono>
using namespace std;

int main() {
    const int SIZE = 100000;
    
    // Vector vs List: insertion performance
    auto start = chrono::high_resolution_clock::now();
    
    vector<int> vec;
    for (int i = 0; i < SIZE; i++) {
        vec.push_back(i);   // Amortized O(1)
    }
    
    auto end = chrono::high_resolution_clock::now();
    auto vecTime = chrono::duration_cast<chrono::milliseconds>(end - start).count();
    
    start = chrono::high_resolution_clock::now();
    
    list<int> lst;
    for (int i = 0; i < SIZE; i++) {
        lst.push_back(i);   // O(1)
    }
    
    end = chrono::high_resolution_clock::now();
    auto listTime = chrono::duration_cast<chrono::milliseconds>(end - start).count();
    
    cout << "=== Performance: push_back " << SIZE << " elements ===" << endl;
    cout << "Vector: " << vecTime << " ms" << endl;
    cout << "List:   " << listTime << " ms" << endl;
    
    // Stack (LIFO) usage
    stack<int> s;
    s.push(1); s.push(2); s.push(3); s.push(4); s.push(5);
    
    cout << "\n=== Stack (LIFO) ===" << endl;
    cout << "Stack contents (top to bottom): ";
    while (!s.empty()) {
        cout << s.top() << " ";
        s.pop();
    }
    cout << endl;
    
    // Queue (FIFO) usage
    queue<int> q;
    q.push(1); q.push(2); q.push(3); q.push(4); q.push(5);
    
    cout << "\n=== Queue (FIFO) ===" << endl;
    cout << "Queue contents (front to back): ";
    while (!q.empty()) {
        cout << q.front() << " ";
        q.pop();
    }
    cout << endl;
    
    return 0;
}

**Output:**

In [ ]:
=== Performance: push_back 100000 elements ===
Vector: 2 ms
List:   16 ms

=== Stack (LIFO) ===
Stack contents (top to bottom): 5 4 3 2 1 

=== Queue (FIFO) ===
Queue contents (front to back): 1 2 3 4 5

---

### Example 3 — Practical Data Processing with STL

In [ ]:
#include <iostream>
#include <vector>
#include <algorithm>
#include <string>
#include <map>
using namespace std;

int main() {
    // Student grades data
    vector<pair<string, int>> grades = {
        {"Arjun", 85},
        {"Priya", 92},
        {"Rahul", 78},
        {"Sneha", 95},
        {"Vikram", 88},
        {"Ananya", 76},
        {"Kiran", 90}
    };
    
    cout << "=== Grade Report ===" << endl;
    cout << "All students:" << endl;
    for (const auto& student : grades) {
        cout << "  " << student.first << ": " << student.second << endl;
    }
    
    // Sort by grade (descending)
    sort(grades.begin(), grades.end(), 
         [](const auto& a, const auto& b) {
             return a.second > b.second;
         });
    
    cout << "\nRanking (by score):" << endl;
    int rank = 1;
    for (const auto& student : grades) {
        cout << "  " << rank++ << ". " << student.first << " (" << student.second << ")" << endl;
    }
    
    // Calculate statistics
    vector<int> scores;
    for (const auto& s : grades) scores.push_back(s.second);
    
    double avg = accumulate(scores.begin(), scores.end(), 0.0) / scores.size();
    auto [minIt, maxIt] = minmax_element(scores.begin(), scores.end());
    
    cout << "\nStatistics:" << endl;
    cout << "  Average: " << avg << endl;
    cout << "  Highest: " << *maxIt << " (" << grades.front().first << ")" << endl;
    cout << "  Lowest: " << *minIt << " (" << grades.back().first << ")" << endl;
    
    // Grade distribution
    map<char, int> distribution;
    for (int s : scores) {
        if (s >= 90) distribution['A']++;
        else if (s >= 80) distribution['B']++;
        else if (s >= 70) distribution['C']++;
        else distribution['D']++;
    }
    
    cout << "\nGrade Distribution:" << endl;
    for (const auto& [grade, count] : distribution) {
        cout << "  " << grade << ": " << count << " student(s)" << endl;
    }
    
    return 0;
}

**Output:**

In [ ]:
=== Grade Report ===
All students:
  Arjun: 85
  Priya: 92
  Rahul: 78
  Sneha: 95
  Vikram: 88
  Ananya: 76
  Kiran: 90

Ranking (by score):
  1. Sneha (95)
  2. Priya (92)
  3. Kiran (90)
  4. Vikram (88)
  5. Arjun (85)
  6. Rahul (78)
  7. Ananya (76)

Statistics:
  Average: 86.2857
  Highest: 95 (Sneha)
  Lowest: 76 (Ananya)

Grade Distribution:
  A: 3 student(s)
  B: 2 student(s)
  C: 2 student(s)

---

## Common Mistakes

| Mistake | Wrong | Correct |
|---|---|---|
| **Mismatching new/delete** | `new int; delete[] ptr;` | Use `delete` for `new`, `delete[]` for `new[]` |
| **Dangling pointer** | `delete ptr;` then using `*ptr` | Set `ptr = nullptr` after delete |
| **Double deletion** | `delete ptr; delete ptr;` | Only delete once; nullify after |
| **Forgetting template syntax** | `class Pair { T first; }` | `template <typename T> class Pair { T first; };` |
| **Using `[]` on vector without bounds check** | `vec[100]` on empty vector | Use `.at(100)` for bounds checking |
| **Modifying container while iterating** | Erasing in a for loop without updating iterator | Use iterator returned by erase |

---

## Interview Questions

1. **What is the difference between `new`/`delete` and `malloc`/`free`?**
   - `new` calls constructor, `delete` calls destructor; `malloc`/`free` only allocate/deallocate memory. `new` returns typed pointer; `malloc` returns `void*`.

2. **What is a memory leak and how do you prevent it?**
   - Memory allocated but never freed. Prevent by: always pairing `new`/`delete`, using RAII, smart pointers (`unique_ptr`, `shared_ptr`).

3. **What are C++ templates?**
   - Templates enable generic programming — writing code that works with any type. **Function templates** for functions, **Class templates** for classes.

4. **What are the main STL containers and when would you use each?**
   - `vector`: dynamic array (random access, fast append/remove at end). `list`: doubly linked list (fast insert/erase anywhere). `stack`: LIFO. `queue`: FIFO. `map`: key-value pairs. `set`: unique sorted elements.

5. **What is the difference between `vector` and `list`?**
   - `vector`: contiguous memory, O(1) random access, O(n) insert in middle. `list`: non-contiguous, O(n) random access, O(1) insert/erase anywhere.

---

## Quick Revision Summary

- **`new`** / **`delete`**: Dynamic memory allocation/deallocation
- **`new[]`** / **`delete[]`**: For arrays — must match!
- **Arrow operator `->`**: Access members through pointers
- **Templates**: `template <typename T>` for generic code
- **`vector`**: Dynamic array — `.push_back()`, `.pop_back()`, `.size()`, `.at()`
- **`list`**: Doubly linked list — `.push_front()`, `.push_back()`, `.sort()`
- **`stack`**: LIFO — `.push()`, `.pop()`, `.top()`
- **`queue`**: FIFO — `.push()`, `.pop()`, `.front()`, `.back()`
- **Iterators**: `begin()`, `end()`, `rbegin()`, `rend()`
- **Algorithms**: `sort()`, `find()`, `count()`, `min_element()`, `max_element()`, `accumulate()`